In [18]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import os
from ultralytics import YOLO
import easyocr
import sys
import os
import re
import torch
os.chdir(r"C:\Users\andre\Desktop\ALL\Master\AIVA_2026-MUVA")
print(torch.cuda.is_available())

True


### Modelo ya entrenado

In [19]:
model = YOLO("system/data/models/best.pt")
reader = easyocr.Reader(['en'], gpu=True)

### Video

In [ ]:
results_video = model("system/data/videos/Video_10.mkv", save=True)

### Video frame a frame

In [20]:
def add_unique(plate_list, plate_text):
    if plate_text and plate_text not in plate_list:
        plate_list.append(plate_text)

In [34]:
cap = cv2.VideoCapture("system/data/videos/Video_10.mkv")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_count = 0
padding_box = 0
plates_verificadas = []
consonantes = "BCDFGHJKLMNPQRSTVWXYZ"

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # Solo procesar cada 2 frames
    if frame_count % 2 == 0:
        print("frame count: ", frame_count)
        results = model(frame)
        result_frame = results[0]
        # Bucle para recortar
        for box, conf in zip(result_frame.boxes.xyxy, result_frame.boxes.conf):
            if conf < 0.50:
                continue
            # Recortar el box
            x1, y1, x2, y2 = map(int, box)
            x1 = max(0, x1 - padding_box)
            y1 = max(0, y1 - padding_box)
            x2 = min(frame.shape[1], x2 + padding_box)
            y2 = min(frame.shape[0], y2 + padding_box)
            plate_crop = frame[y1:y2, x1:x2]
            # Preprocesado para OCR
            gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
            gray = cv2.resize(gray, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            gray = clahe.apply(gray)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            # Aplicar OCR
            result_text = reader.readtext(thresh, detail=1)
            texts = [f"{text}({conf:.2f})" for bbox, text, conf in result_text][::-1]
            print("@@@@@ Matricula @@@@@@")
            print(" ".join(texts))
            # Comprobacion de formato matriula
            if len(result_text) == 1:
                text, conf = result_text[0][1], result_text[0][2]
                if conf > 0.70:
                    text = text.strip().upper()[-8:]
                    if (
                        len(text) == 8 and
                        text[:4].isdigit() and
                        text[4] == " " and
                        len(text[5:]) == 3 and
                        all(c in consonantes for c in text[5:])
                    ):
                        add_unique(plates_verificadas, text)
            elif len(result_text) == 2:
                text1, conf1 = result_text[0][1], result_text[0][2]
                text2, conf2 = result_text[1][1], result_text[1][2]
                if conf1 > 0.70 and conf2 > 0.70:
                    # Solo numeros y 4 digitos
                    text2 = re.sub(r'[^0-9]', '', text2)[-4:]
                    # Solo consonantes y 3 digitos
                    text1 = ''.join(c for c in re.sub(r'[^A-Z]', '', text1.upper()) if c not in "AEIOU")[-3:]
                    if len(text2) == 4 and len(text1) == 3:
                        add_unique(plates_verificadas, text2 + " " + text1)
    frame_count += 1

cap.release()
cv2.destroyAllWindows()

frame count:  0

0: 384x640 1 ['plate'], 275.9ms
Speed: 2.7ms preprocess, 275.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)
frame count:  2

0: 384x640 2 ['plate']s, 71.1ms
Speed: 2.5ms preprocess, 71.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
6638(0.61) GLN(0.59)
frame count:  4

0: 384x640 1 ['plate'], 18.9ms
Speed: 1.4ms preprocess, 18.9ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
4638(0.94) GLN(0.49)
frame count:  6

0: 384x640 1 ['plate'], 13.6ms
Speed: 1.5ms preprocess, 13.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
014638(0.40)
frame count:  8

0: 384x640 1 ['plate'], 9.5ms
Speed: 1.4ms preprocess, 9.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
@@@@@ Matricula @@@@@@
4638(0.93) GLN(0.98)
frame count:  10

0: 384x640 1 ['plate'], 9.4ms
Speed: 2.3ms preprocess, 9.4ms inference, 1.2ms postproces